# Exercise 01: The Jaffle Shop

You've been brought in as the first data engineer at **The Jaffle Shop**, a fast-growing food delivery business.
They have raw transactional data sitting in CSV files and need you to build a proper data warehouse.

Your job is to transform the raw source data into clean, analytics-ready tables by following the same **layered architecture** used in dbt projects.

---

### The architecture you'll build

```
SOURCE DATA (CSV files)
  data/raw_customers.csv
  data/raw_orders.csv
  data/raw_payments.csv
         │
         ▼
STAGING LAYER  — clean and rename columns, one table per source
  stg_customers
  stg_orders
  stg_payments
         │
         ▼
MARTS LAYER  — business-level tables built from the staging layer
  fct_orders          (one row per order, with payment totals)
  dim_customers       (one row per customer, with lifetime metrics)
  monthly_jaffle_metrics  (aggregated monthly summary)
```

**Why two layers?**
- The staging layer is a 1-to-1 mapping of source data — rename columns, fix types, nothing else.
- The marts layer combines and aggregates staging tables to answer business questions.
- This separation makes it easy to change your source without breaking downstream tables.

## Setup

Run this cell first. It connects to DuckDB and loads the three raw CSV files as tables.

In [ ]:
import duckdb

con = duckdb.connect()

# Load source CSV files into raw tables
con.execute('''
    CREATE OR REPLACE TABLE raw_customers AS
    SELECT * FROM read_csv_auto('../data/raw_customers.csv')
''')

con.execute('''
    CREATE OR REPLACE TABLE raw_orders AS
    SELECT * FROM read_csv_auto('../data/raw_orders.csv')
''')

con.execute('''
    CREATE OR REPLACE TABLE raw_payments AS
    SELECT * FROM read_csv_auto('../data/raw_payments.csv')
''')

print('Setup complete. Raw tables loaded.')

## Step 1: Preview the Source Data

Before writing any transformations, understand what you're working with.
Run each cell below to see the raw source tables.

In [ ]:
# raw_customers — one row per customer
con.execute('SELECT * FROM raw_customers LIMIT 5').df()

In [ ]:
# raw_orders — one row per order
con.execute('SELECT * FROM raw_orders LIMIT 5').df()

In [ ]:
# raw_payments — one row per payment attempt (an order can have multiple payments)
# Note: column names are UPPERCASE in this source
# Note: AMOUNT is in cents (divide by 100 to get dollars)
con.execute('SELECT * FROM raw_payments LIMIT 5').df()

---
## Staging Layer

The staging layer's job is simple: **clean up the source data**.
- Rename columns to a consistent naming convention (`snake_case`)
- Cast types where needed
- No joins, no aggregations — keep it one-to-one with the source

Each staging table is prefixed with `stg_`.

### Task 1: `stg_customers`

Create a staging table for customers by renaming columns from the raw source.

**Output columns:**
| Column | Source column | Notes |
|---|---|---|
| `customer_id` | `id` | Rename |
| `first_name` | `first_name` | Keep as-is |
| `last_name` | `last_name` | Keep as-is |

In [ ]:
con.execute('''
    CREATE OR REPLACE TABLE stg_customers AS
    SELECT
        -- ✏️ Write your SQL here
        -- Rename 'id' to 'customer_id', keep first_name and last_name

    FROM raw_customers
''')

In [ ]:
# Check your work — should have columns: customer_id, first_name, last_name
con.execute('SELECT * FROM stg_customers LIMIT 5').df()

### Task 2: `stg_orders`

Create a staging table for orders.

**Output columns:**
| Column | Source column | Notes |
|---|---|---|
| `order_id` | `id` | Rename |
| `customer_id` | `user_id` | Rename |
| `order_date` | `order_date` | Keep as-is |
| `status` | `status` | Keep as-is |

In [ ]:
con.execute('''
    CREATE OR REPLACE TABLE stg_orders AS
    SELECT
        -- ✏️ Write your SQL here
        -- Rename 'id' to 'order_id' and 'user_id' to 'customer_id'

    FROM raw_orders
''')

In [ ]:
# Check your work — should have columns: order_id, customer_id, order_date, status
con.execute('SELECT * FROM stg_orders LIMIT 5').df()

### Task 3: `stg_payments`

Create a staging table for payments. This one needs some extra work — the source columns are UPPERCASE and the amounts are in **cents** (divide by 100 to get dollars).

**Output columns:**
| Column | Source column | Notes |
|---|---|---|
| `payment_id` | `ID` | Rename |
| `order_id` | `ORDERID` | Rename |
| `payment_method` | `PAYMENTMETHOD` | Rename |
| `status` | `STATUS` | Rename |
| `amount` | `AMOUNT` | Rename AND divide by 100.0 |
| `payment_date` | `CREATED` | Rename |

In [ ]:
con.execute('''
    CREATE OR REPLACE TABLE stg_payments AS
    SELECT
        -- ✏️ Write your SQL here
        -- Hint: DuckDB column names are case-insensitive, so ORDERID works as orderid
        -- Don't forget to divide AMOUNT by 100.0

    FROM raw_payments
''')

In [ ]:
# Check your work — should have 6 columns with lowercase names, amount in dollars
con.execute('SELECT * FROM stg_payments LIMIT 5').df()

---
## Marts Layer

Now that the staging layer is clean, we can build the business-level tables.
These tables answer real questions: *How much did each customer spend? What were our revenues this month?*

- `fct_` prefix = **fact tables** — one row per business event (an order, a sale, a payment)
- `dim_` prefix = **dimension tables** — one row per entity (a customer, a product, a location)

### Task 4: `fct_orders`

Create a fact table with **one row per order**, including the total payment amounts broken out by payment method.

You'll need to JOIN `stg_orders` with `stg_payments` and aggregate the payment amounts.
Only count payments where `status = 'success'`.

**Output columns:**
| Column | Where it comes from |
|---|---|
| `order_id` | `stg_orders.order_id` |
| `customer_id` | `stg_orders.customer_id` |
| `order_date` | `stg_orders.order_date` |
| `status` | `stg_orders.status` |
| `credit_card_amount` | SUM of successful credit_card payments |
| `coupon_amount` | SUM of successful coupon payments |
| `bank_transfer_amount` | SUM of successful bank_transfer payments |
| `gift_card_amount` | SUM of successful gift_card payments |
| `amount` | Total of all successful payments |

**Hint:** Use a `CASE WHEN` inside `SUM()` to pivot payment methods into columns:
```sql
SUM(CASE WHEN payment_method = 'credit_card' AND status = 'success' THEN amount ELSE 0 END) AS credit_card_amount
```

In [ ]:
con.execute('''
    CREATE OR REPLACE TABLE fct_orders AS
    SELECT
        -- ✏️ Write your SQL here
        -- Join stg_orders with stg_payments on order_id
        -- Use LEFT JOIN so orders with no payments still appear
        -- Use COALESCE(..., 0) to turn NULLs into 0
        -- GROUP BY the order columns (not the payment columns)

    FROM stg_orders o
    LEFT JOIN stg_payments p ON o.order_id = p.order_id
    GROUP BY
        -- ✏️ List the columns you're not aggregating
''')

In [ ]:
# Check your work — should have 99 rows (one per order), with payment amounts in dollars
print('Row count:', con.execute('SELECT COUNT(*) FROM fct_orders').fetchone()[0])
con.execute('SELECT * FROM fct_orders LIMIT 5').df()

### Task 5: `dim_customers`

Create a dimension table with **one row per customer**, enriched with their order history.

You'll join `stg_customers` with `fct_orders` and aggregate order metrics per customer.
Use a `LEFT JOIN` so customers with no orders still appear (with NULLs or 0s for metrics).

**Output columns:**
| Column | Where it comes from |
|---|---|
| `customer_id` | `stg_customers.customer_id` |
| `first_name` | `stg_customers.first_name` |
| `last_name` | `stg_customers.last_name` |
| `first_order_date` | MIN of `fct_orders.order_date` |
| `most_recent_order_date` | MAX of `fct_orders.order_date` |
| `number_of_orders` | COUNT of `fct_orders.order_id` |
| `customer_lifetime_value` | SUM of `fct_orders.amount` |

In [ ]:
con.execute('''
    CREATE OR REPLACE TABLE dim_customers AS
    SELECT
        -- ✏️ Write your SQL here
        -- Hint: use COALESCE(SUM(o.amount), 0) so customers with no orders show 0

    FROM stg_customers c
    LEFT JOIN fct_orders o ON c.customer_id = o.customer_id
    GROUP BY
        -- ✏️ List the non-aggregated columns
''')

In [ ]:
# Check your work — should have 100 rows (one per customer)
# Customers who never ordered should have NULL for dates and 0 for number_of_orders
print('Row count:', con.execute('SELECT COUNT(*) FROM dim_customers').fetchone()[0])
con.execute('''
    SELECT * FROM dim_customers
    ORDER BY customer_lifetime_value DESC
    LIMIT 10
''').df()

### Task 6: `monthly_jaffle_metrics`

Create a monthly summary table from `fct_orders`.

**Output columns:**
| Column | How to calculate |
|---|---|
| `month` | Year-month string from `order_date`, e.g. `'2018-01'` |
| `num_orders` | COUNT of orders |
| `num_customers` | COUNT DISTINCT of customers who ordered |
| `total_revenue` | SUM of `amount` |
| `returned_revenue` | SUM of `amount` where status is `'returned'` or `'return_pending'` |

**Hint:** Use `strftime(order_date, '%Y-%m')` to format a date as year-month.

In [ ]:
con.execute('''
    CREATE OR REPLACE TABLE monthly_jaffle_metrics AS
    SELECT
        -- ✏️ Write your SQL here
        -- Hint: strftime(order_date, '%Y-%m') formats a date as '2018-01'
        -- Hint: use CASE WHEN inside SUM() for returned_revenue

    FROM fct_orders
    GROUP BY
        -- ✏️
    ORDER BY
        -- ✏️
''')

In [ ]:
# Check your work — should have one row per month, ordered chronologically
con.execute('SELECT * FROM monthly_jaffle_metrics').df()

---
## All done! Verify your database

Run the cell below to see all the tables you've built and confirm the row counts.

In [ ]:
# All tables in the database
con.execute('SHOW TABLES').df()

In [ ]:
# Quick sanity check on row counts
checks = [
    ('raw_customers',          100),
    ('raw_orders',              99),
    ('stg_customers',          100),
    ('stg_orders',              99),
    ('fct_orders',              99),
    ('dim_customers',          100),
]

for table, expected in checks:
    actual = con.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    status = '✅' if actual == expected else '❌'
    print(f'{status}  {table}: {actual} rows (expected {expected})')